In [6]:
%load_ext autoreload
%autoreload 2

import os

SELECTED_DEVICE = '9'
print(f'Setting CUDA visible devices to [{SELECTED_DEVICE}]')
os.environ['CUDA_VISIBLE_DEVICES'] = f'{SELECTED_DEVICE}'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setting CUDA visible devices to [9]


In [7]:
import matplotlib.pyplot as plt

import jax
jax.config.update("jax_enable_x64", True)

import jax.random as jrnd
import jax.numpy as jnp
from jax.scipy.special import gammaln, logsumexp
import blackjax
import numpyro as npr
import numpyro.distributions as dist
import numpyro.distributions.transforms as nprb

import sys
import time

from bamojax.base import Model
from bamojax.samplers import mcmc_sampler
from bamojax.inference import SMCInference, MCMCInference

from bamojax.model_search.anchored_rjsmc import RJSMCInference, get_model_samples


print('Python version:     ', sys.version)
print('Jax version:        ', jax.__version__)
print('BlackJax version:   ', blackjax.__version__)
print('Numpyro version:    ', npr.__version__)
print('Jax default backend:', jax.default_backend())
print('Jax devices:        ', jax.devices())

SMALL_SIZE = 12
MEDIUM_SIZE = 16
BIGGER_SIZE = 18

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=MEDIUM_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

Python version:      3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 11.4.0]
Jax version:         0.10.2
BlackJax version:    1.7.1
Numpyro version:     0.22.0
Jax default backend: gpu
Jax devices:         [CudaDevice(id=0)]


Simulate data

In [8]:
key = jrnd.PRNGKey(1234)

n, a0, b0, g = 40, 3.0, 2.0, 1.0
kx, ke = jrnd.split(key)
X = jrnd.normal(kx, (n, 3))
y = 0.45 * X[:, 0] + 0.25 * X[:, 1] + jrnd.normal(ke, (n,))

In [9]:
def make_model(cols):
    """Include different subsets of the predictors
    
    """

    m = Model(f'cols={cols}')
    s2 = m.add_node('sigma2', distribution=dist.InverseGamma(a0, b0))
    if len(cols) == 0:
        m.add_node('y', observations=y, distribution=dist.Normal,
                   parents=dict(sigma2=s2), link_fn=lambda sigma2: dict(loc=0.0, scale=jnp.sqrt(sigma2)))
    else:
        Xc = X[:, cols]
        beta = m.add_node('beta', distribution=dist.Normal, shape=(len(cols),), parents=dict(sigma2=s2),
                          link_fn=lambda sigma2: dict(loc=0.0, scale=jnp.sqrt(g * sigma2)))
        m.add_node('y', observations=y, distribution=dist.Normal, parents=dict(beta=beta, sigma2=s2, X=Xc),
                   link_fn=lambda beta, sigma2, X: dict(loc=X @ beta, scale=jnp.sqrt(sigma2)))
    return m

#

def exact_lml(cols):
    """ Long live the Gaussian! Exact lmls:

    """
    Xc = X[:, cols]
    S = jnp.eye(n) + g * Xc @ Xc.T
    nu = 2 * a0
    _, logdet = jnp.linalg.slogdet((b0 / a0) * S)
    q = y @ jnp.linalg.solve((b0 / a0) * S, y)
    return gammaln((nu + n) / 2) - gammaln(nu / 2) - n / 2 * jnp.log(nu * jnp.pi) - 0.5 * logdet - (nu + n) / 2 * jnp.log1p(q / nu)

In [10]:
subsets = [[], [0], [0, 1], [0, 1, 2]]
models = [make_model(c) for c in subsets]
lmls = jnp.array([exact_lml(c) for c in subsets])
exact_probs = jax.nn.softmax(lmls)

print('exact model probs:', jnp.round(exact_probs, 3), ' exact log p(y):', logsumexp(lmls) - jnp.log(4))
print('exact lml per model:', jnp.round(lmls, 2))

# sanity check of the bamojax densities against the closed form (prior-sampling estimate not needed: check logpdf at a point)
kernels = [dict(mcmc_kernel=blackjax.normal_random_walk, mcmc_parameters=dict(sigma=0.15 * jnp.eye(int(m.get_model_size())))) for m in models]

for label, kw in [('RJ only        ', dict()), ('RJ + 5 RMH steps', dict(within_model_kernels=kernels, num_mutations=5))]:
    engine = RJSMCInference(models, num_particles=2000, **kw)
    run = jax.jit(engine.run)
    t0 = time.time(); res = run(jrnd.PRNGKey(0)); jax.block_until_ready(res['lml']); t_compile = time.time() - t0
    probs, lml_all, accs, ts = [], [], [], []
    for s in range(1, 9):
        t0 = time.time(); res = run(jrnd.PRNGKey(s)); jax.block_until_ready(res['lml']); ts.append(time.time() - t0)
        probs.append(res['model_probs']); lml_all.append(res['lml']); accs.append(res['rj_acceptance_rate'])
    probs = jnp.stack(probs)
    print(f'\n[{label}] iters={int(res["n_iter"])}  first call {t_compile:.1f}s, then {jnp.mean(jnp.array(ts)):.2f}s/run')
    print('  model probs mean:', jnp.round(probs.mean(0), 3), ' sd:', jnp.round(probs.std(0), 3))
    print('  log p(y) mean/sd:', float(jnp.mean(jnp.array(lml_all))), float(jnp.std(jnp.array(lml_all))))
    print('  lml per model (last run):', jnp.round(res['lml_per_model'], 2))
    print('  RJ cross-model acceptance:', float(jnp.mean(jnp.array(accs))))
    s = get_model_samples(res, 2)
    print('  model 2 posterior mean beta:', jnp.round(s['beta'].mean(0), 3), ' n =', s['beta'].shape[0])



exact model probs: [0.007 0.005 0.847 0.141]  exact log p(y): -62.039512586731355
exact lml per model: [-65.61 -65.98 -60.82 -62.61]

[RJ only        ] iters=47  first call 3.5s, then 0.49s/run
  model probs mean: [0.008 0.006 0.844 0.142]  sd: [0.002 0.002 0.008 0.008]
  log p(y) mean/sd: -62.056181843553226 0.016264406339293925
  lml per model (last run): [-65.7  -66.31 -60.81 -62.68]
  RJ cross-model acceptance: 0.5705576863214655
  model 2 posterior mean beta: [0.397 0.705]  n = 1716

[RJ + 5 RMH steps] iters=20  first call 4.8s, then 0.25s/run
  model probs mean: [0.008 0.006 0.837 0.149]  sd: [0.001 0.002 0.008 0.007]
  log p(y) mean/sd: -62.051809883156494 0.021907578418597185
  lml per model (last run): [-65.3  -65.85 -60.81 -62.59]
  RJ cross-model acceptance: 0.6483544269770237
  model 2 posterior mean beta: [0.4   0.717]  n = 1684


In [11]:
# exact posterior mean of beta in model 2 for reference
Xc = X[:, subsets[2]]
print('\nexact post. mean beta, model 2:', jnp.round(jnp.linalg.solve(Xc.T @ Xc + jnp.eye(2) / g, Xc.T @ y), 3))

# multiple chains via the InferenceEngine vmap
eng = RJSMCInference(models, num_particles=1000, num_chains=4)
r = eng.run(jrnd.PRNGKey(42))
print('num_chains=4 model_probs shape:', r['model_probs'].shape)


exact post. mean beta, model 2: [0.396 0.711]
num_chains=4 model_probs shape: (4, 4)
